# Этап 3: selective direction, альтернативные горизонты и neutral-класс

Цель — проверить, можно ли повысить точность направления abnormal return без тяжёлой новой модели: отвечать только на уверенных событиях, изменить горизонт результата или явно выделить нейтральные движения.

## План и заранее заданный критерий успеха

Проверяем горизонты 30/60/120/240 минут, selective coverage 20/30/50/100% и neutral-зоны ±0,10/0,25/0,50%. Во всех вариантах используется фиксированный `reaction_core` этапа 2. Порог ответа выбирается только на validation предыдущих месяцев. Evaluation неизменен: 480 событий, семь walk-forward folds, два месяца validation, embargo 72 часа, группы повторов не разделяются. Конфигурация считается успешной только при hit rate ≥58%, coverage ≥25%, нижней границе cluster-bootstrap 95% CI >50% и результате >50% минимум в 5/7 folds.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
experiment_dir = (
    cwd
    if (cwd / 'run_experiment.py').exists()
    else cwd / 'experiments' / 'direction_stage3'
)
repo_root = experiment_dir.parents[1]
sys.path.insert(0, str(repo_root))
from experiments.direction_stage3.run_experiment import run_experiment  # noqa: E402
from experiments.finbert_stage1.run_experiment import sha256_file  # noqa: E402
from experiments.paths import dataset_path_from_environment, stage_artifact_directory  # noqa: E402

dataset_path = dataset_path_from_environment()
artifact_dir = stage_artifact_directory('direction_stage3')
market_features_path = (
    stage_artifact_directory('market_stage2')
    / 'market_features.csv'
)
labels_path = artifact_dir / 'horizon_labels.csv'
print(dataset_path)

In [ ]:
# Читаем зафиксированные артефакты. Для offline-пересчёта: RUN_STAGE3=1.
if os.environ.get('RUN_STAGE3') == '1':
    result = run_experiment(
        dataset_path,
        artifact_dir,
        market_features_path=market_features_path,
        labels_path=labels_path,
    )
else:
    result = json.loads((artifact_dir / 'metrics.json').read_text())
label_report = json.loads((artifact_dir / 'label_build_report.json').read_text())
print(result['experiment'], result['created_at'])

## Автоматический контроль корректности

In [ ]:
selective = pd.read_csv(artifact_dir / 'selective_predictions.csv')
ternary = pd.read_csv(artifact_dir / 'ternary_predictions.csv')
labels = pd.read_csv(labels_path)

assert result['dataset']['rows'] == 1238
assert len(selective) == 4 * 480
assert selective.groupby('horizon_minutes').size().eq(480).all()
assert selective.groupby('horizon_minutes').fold.nunique().eq(7).all()
assert len(ternary) == 4 * 3 * 480
assert ternary.groupby(['horizon_minutes', 'neutral_threshold_pct']).size().eq(480).all()
assert result['passing_selective_configurations'] == []
assert result['passing_ternary_configurations'] == []
assert label_report['reproduction_240m']['derived_240m_rows'] == 1237
assert label_report['reproduction_240m']['entry_price_max_abs_error'] == 0
assert label_report['reproduction_240m']['stock_max_abs_error_pct'] == 0
assert label_report['reproduction_240m']['benchmark_max_abs_error_pct'] == 0
assert label_report['reproduction_240m']['abnormal_max_abs_error_pct'] == 0
for filename, expected_hash in result['artifact_hashes'].items():
    assert sha256_file(artifact_dir / filename) == expected_hash
print(
    'QA passed: exact 4h reproduction, 480 rows × 4 horizons, '
    '7 folds, artifact hashes, no passing configurations.'
)

## Selective direction: risk–coverage

In [ ]:
rows = []
for horizon, horizon_metrics in result['selective_binary'].items():
    for coverage, metrics in horizon_metrics['coverage_points'].items():
        rows.append({
            'horizon, min': int(horizon),
            'target coverage': int(coverage) / 100,
            'actual coverage': metrics['actual_coverage'],
            'N': metrics['n'],
            'hit rate': metrics['hit_rate'],
            'cluster CI low': metrics['cluster_bootstrap']['ci95_low'],
            'cluster CI high': metrics['cluster_bootstrap']['ci95_high'],
            'positive folds': metrics['positive_folds'],
        })
selective_table = pd.DataFrame(rows).sort_values(['horizon, min', 'target coverage'])
display(selective_table.round(4))
display(Image(filename=str(artifact_dir / 'risk_coverage.png')))

## Трёхклассовая постановка

In [ ]:
rows = []
for key, metrics in result['ternary'].items():
    rows.append({
        'configuration': key,
        'directional coverage': metrics['predicted_directional_coverage'],
        'directional precision': metrics['directional_precision'],
        'cluster CI low': metrics['directional_cluster_bootstrap']['ci95_low'],
        'cluster CI high': metrics['directional_cluster_bootstrap']['ci95_high'],
        'macro-F1': metrics['macro_f1'],
        'positive folds': metrics['positive_folds'],
    })
ternary_table = pd.DataFrame(rows).sort_values('directional precision', ascending=False)
display(ternary_table.round(4))

## Config 6 на альтернативных горизонтах

In [ ]:
rows = []
for horizon, metrics in result['config6_by_horizon'].items():
    rows.append({
        'horizon, min': int(horizon),
        'N': metrics['n'],
        'hit rate': metrics['hit_rate'],
        'balanced accuracy': metrics['balanced_accuracy'],
        'cluster CI low': metrics['cluster_bootstrap']['ci95_low'],
        'cluster CI high': metrics['cluster_bootstrap']['ci95_high'],
        'positive folds': metrics['positive_folds'],
    })
display(pd.DataFrame(rows).sort_values('horizon, min').round(4))

## Вывод

Ни selective abstention, ни neutral-класс, ни замена горизонта не дают подтверждённого направления. Лучший practically relevant selective-вариант — 52,68% при coverage 42,71%, но cluster CI [46,19%; 59,33%] включает 50% и только 3/7 folds положительны. У config 6 есть exploratory-пик 55,41% на 120 минутах, однако cluster CI [47,13%; 63,75%] также включает 50%. Эти варианты не следует переносить в продукт; следующий дешёвый путь — структурированные surprise/event-type/sector/regime признаки этапа 4.